# 05 · EDA avanzado, clusters, anomalías y predicción

> **Estado de evidencia:** los CSV suministrados son plantillas sin observaciones. Las celdas calculan resultados únicamente si se cargan extractos reales. No se fabrican hallazgos. Las fórmulas constituyen especificaciones metodológicas sujetas a aprobación del contrato de métricas.

**Convenciones:** $i$ crédito, $c$ cohorte de originación, $t$ corte calendario, $m$ meses desde originación (MOB), $D_{i,t}$ días de mora, $B_{i,t}$ saldo capital, $L_{i,t}$ cupo. Los límites de ventanas y las reglas de castigo deben parametrizarse.

## 1. Objetivo, alcance y decisiones

Se distinguen observaciones a nivel crédito-corte, crédito-MOB, evento y cliente. Se prohíben joins uno-a-muchos sin agregación previa. La trazabilidad debe conservar fecha de vigencia y disponibilidad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE = Path.cwd()
if not (BASE/'plantilla_panel_mensual.csv').exists():
    BASE = Path('/mnt/data/estudio_riesgo_credito')
assert BASE.exists(), 'Ubique el notebook junto a las plantillas CSV'
plt.rcParams.update({'figure.figsize':(9,4.8),'axes.grid':True,'grid.alpha':.2})
def load(name, parse_dates=None):
    path=BASE/name
    df=pd.read_csv(path,parse_dates=parse_dates or [])
    print(f'{name}: {len(df):,} registros, {len(df.columns)} columnas')
    return df

## EDA robusto y dependencias

Para numéricas, evaluar cuantiles, MAD, correlación de Spearman y asociaciones condicionales. La correlación de Spearman es $\rho_S=\mathrm{Corr}(F_X(X),F_Y(Y))$ bajo continuidad; captura monotonía, no causalidad. Evaluar estabilidad por cohorte y MOB; imputaciones y escalados se ajustan **solo con entrenamiento**.

In [ ]:
df=load('plantilla_panel_mensual.csv')
if not df.empty:
    nums=df.select_dtypes(include='number')
    display(nums.describe(percentiles=[.01,.1,.5,.9,.99]).T.head(20))
    if nums.shape[1]>1:
        corr=nums.corr(method='spearman');plt.imshow(corr, vmin=-1,vmax=1);plt.xticks(range(len(corr)),corr.columns,rotation=90);plt.yticks(range(len(corr)),corr.columns);plt.colorbar(label='Spearman');plt.title('Dependencia monotónica');plt.tight_layout();plt.show()
else: print('PENDIENTE: EDA real')

## Clustering y validación

Para observaciones estandarizadas $z_i$, K-means minimiza:
$$\min_{\{C_k,\mu_k\}}\sum_{k=1}^K\sum_{i\in C_k}\|z_i-\mu_k\|_2^2.$$
Variables mixtas requieren distancia de Gower, FAMD o K-Prototypes. Comparar estabilidad temporal, tamaño mínimo, silhouette y interpretabilidad; **UMAP es visualización, no evidencia de grupos naturales**.

In [ ]:
if not df.empty:
    cols=[c for c in ['mora','saldo_capital','cupo_vigente'] if c in df]
    x=df[cols].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
    if len(x)>=30 and len(cols)>=2:
        try:
            from sklearn.preprocessing import StandardScaler
            from sklearn.cluster import KMeans
            z=StandardScaler().fit_transform(x)
            km=KMeans(n_clusters=min(4,max(2,len(x)//20)),random_state=42,n_init=10).fit(z)
            print('Tamaños clusters:',pd.Series(km.labels_).value_counts().sort_index().to_dict())
            plt.scatter(z[:,0],z[:,1],c=km.labels_,s=8,alpha=.5);plt.title('Clustering exploratorio (variables de ejemplo)');plt.xlabel(cols[0]);plt.ylabel(cols[1]);plt.tight_layout();plt.show()
        except ImportError:print('Instalar scikit-learn para clustering')
    else:print('Datos insuficientes para clustering')
else:print('PENDIENTE: clustering')

## Anomalías: robustez y falsos positivos

Una puntuación robusta univariada es $z_i^{MAD}=0.6745(x_i-\operatorname{mediana}(x))/MAD(x)$, si $MAD>0$. Isolation Forest usa longitudes cortas de aislamiento para señalar rareza; no clasifica automáticamente fraude. Validar precisión en el top-$k$, estabilidad por segmento y revisión humana.

In [ ]:
if not df.empty and 'saldo_capital' in df:
    x=pd.to_numeric(df.saldo_capital,errors='coerce').dropna()
    med=x.median();mad=(x-med).abs().median()
    if mad>0:
        score=.6745*(x-med)/mad
        print('Observaciones con |z_MAD|>3.5:',int(score.abs().gt(3.5).sum()))
    else:print('MAD=0; método no aplicable sin adaptación')
else:print('PENDIENTE: anomalías')

## Modelo de originación: probabilidad de EVER30 M3

Objetivo $Y_i=\mathbf1(\max_{m\le3}DPD_{i,m}\ge30)$ en cohortes maduras. Modelo base:
$$P(Y_i=1\mid X_{i,0})=\sigma(\beta_0+X_{i,0}^{\top}\beta),\quad \sigma(z)=\frac1{1+e^{-z}}.$$
Pérdida logarítmica $\mathcal L=-N^{-1}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]$. **Solo** variables disponibles en originación; no usar DPD, pagos ni avances futuros.

In [ ]:
origin=load('plantilla_creditos_originacion.csv')
print('Campos originación:',list(origin.columns))
if origin.empty:print('PENDIENTE: target maduro + predictores T0 + split temporal')

## Supervivencia, censura y alertas tempranas

Tiempo hasta primer 30+ $T_i$, censura $C_i$: $\tilde T_i=\min(T_i,C_i)$ y $\delta_i=\mathbf1(T_i\le C_i)$. Kaplan-Meier:
$$\widehat S(t)=\prod_{u\le t}\left(1-\frac{d_u}{n_u}\right).$$
Para alerta en MOB1, usar variables observadas hasta MOB1 y target futuro condicionado a **no haber presentado ya el evento**; evita redefinir riesgo de ocurrencia inicial como recurrencia.

In [ ]:
if not df.empty and {'mob','mora','loan_key'}.issubset(df):
    print('MOB disponibles:',sorted(pd.to_numeric(df.mob,errors='coerce').dropna().unique())[:24])
else:print('PENDIENTE: cohortes con censura identificable')

## Validación temporal, calibración y drift

Usar partición cronológica por fecha de originación y separación por cliente cuando haya múltiples créditos. Brier:
$$BS=\frac1N\sum_i(p_i-y_i)^2.$$
Calibración: comparar $E[Y\mid \hat p\in I_b]$ contra $E[\hat p\mid\hat p\in I_b]$. Reportar AUC, PR-AUC, KS, lift, intervalos de confianza por bootstrap agrupado, drift PSI y estabilidad por cohortes/segmentos. Evitar *random split* que mezcle observaciones del mismo cliente o periodos.

In [ ]:
print('Plan de validación: train cohortes antiguas; validación intermedia; test cohortes recientes maduras.')
print('Sin etiquetas observadas y fechas no se estiman AUC, KS, Brier ni calibración.')

## Contrafactual y sesgo de aprobación

Los resultados observados corresponden a solicitudes aprobadas: $P(Y\mid X,A=1)$ no identifica necesariamente $P(Y\mid X)$ en solicitantes rechazados. Propensity weighting requiere positividad e ignorabilidad condicional:
$$\widehat E[Y(a)]=\frac{1}{N}\sum_i\frac{\mathbf1(A_i=a)Y_i}{\widehat P(A_i=a\mid X_i)}.$$
No utilizar este estimador sin observar outcomes comparables y verificar supuestos. Las simulaciones de cambio de política no son inferencias causales por defecto.

In [ ]:
print('PENDIENTE: historial de solicitudes aprobadas/rechazadas y outcomes observables para evaluación de selección')

## Limitaciones y siguiente ejecución

**Sin datos reales** no se pueden inferir tasas, significancia, calidad predictiva ni relaciones causales. Cargar las plantillas desde Athena con llaves y fechas; verificar los resultados de los controles antes de interpretar gráficos. Para producción: pruebas temporales, monitoreo de sesgo de selección, documentación de fuentes y revisión de privacidad.